In [4]:
import pandas as pd
df = pd.read_csv("2019-Dec.csv", nrows=100000)
df.head()

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-12-01 00:00:00 UTC,remove_from_cart,5712790,1487580005268456287,NaN,f.o.x,6.27,576802932,51d85cb0-897f-48d2-918b-ad63965c12dc
1,2019-12-01 00:00:00 UTC,view,5764655,1487580005411062629,NaN,cnd,29.05,412120092,8adff31e-2051-4894-9758-224bfa8aec18
2,2019-12-01 00:00:02 UTC,cart,4958,1487580009471148064,NaN,runail,1.19,494077766,c99a50e8-2fac-4c4d-89ec-41c05f114554
3,2019-12-01 00:00:05 UTC,view,5848413,1487580007675986893,NaN,freedecor,0.79,348405118,722ffea5-73c0-4924-8e8f-371ff8031af4
4,2019-12-01 00:00:07 UTC,view,5824148,1487580005511725929,NaN,NaN,5.56,576005683,28172809-7e4a-45ce-bab0-5efa90117cd5


In [2]:
df["event_type"].value_counts()

event_type
view                46007
cart                26771
remove_from_cart    21350
purchase             5872
Name: count, dtype: int64

In [5]:
import sqlite3
import pandas as pd
conn = sqlite3.connect("ecommerce.db")
df.to_sql("events", conn, if_exists="replace", index=False)

100000

In [6]:
query = """
SELECT 
    event_type,
    COUNT(DISTINCT user_id) as users
FROM events
GROUP BY event_type
"""

pd.read_sql(query, conn)

,event_type,users
0,cart,3315
1,purchase,644
2,remove_from_cart,1973
3,view,13382


In [7]:
query = """
SELECT 
    event_type,
    COUNT(DISTINCT user_id) as users
FROM events
GROUP BY event_type
"""

funnel = pd.read_sql(query, conn)
funnel

,event_type,users
0,cart,3315
1,purchase,644
2,remove_from_cart,1973
3,view,13382


In [8]:
view_users = funnel[funnel["event_type"]=="view"]["users"].values[0]
cart_users = funnel[funnel["event_type"]=="cart"]["users"].values[0]
purchase_users = funnel[funnel["event_type"]=="purchase"]["users"].values[0]
view_to_cart = cart_users / view_users
cart_to_purchase = purchase_users / cart_users
print("View → Cart:", view_to_cart)
print("Cart → Purchase:", cart_to_purchase)

View → Cart: 0.24772081901061127
Cart → Purchase: 0.1942684766214178
